# 02 — Player seasons

Build the clean, consolidated `player_season` table (2011-12 → 2025-26) from the cached raw responses in `data/raw/`.

## Step 2A — Raw data inventory

`python -m unicorn.download` populates `data/raw/` (703 requests, resumable). Re-running it here costs nothing because every request is already cached. Before any cleaning, check the raw layer is complete and plausible.

In [1]:
import pandas as pd
import numpy as np
from unicorn.download import download_all

log = download_all()   # instant if already downloaded
print(log["status"].str.split(":").str[0].value_counts().to_dict())
log[log["status"].str.startswith("FAILED")]

{'cached': 703}


,endpoint,season,season_type_all_star,measure_type_detailed_defense,per_mode_detailed,rows,status,distance_range,per_mode_simple,team_id,season_all_time


In [2]:
# Rows per season and endpoint: player counts should be stable (~450-600 regular season)
lds = log[log["endpoint"] == "LeagueDashPlayerStats"]
lds.pivot_table(index="season", columns=["season_type_all_star", "measure_type_detailed_defense"],
                values="rows", aggfunc="first")

season_type_all_star          Playoffs                                  \
measure_type_detailed_defense Advanced Base Defense Misc Scoring Usage   
season                                                                   
2011-12                            204  204     204  204     204   204   
2012-13                            209  209     209  209     209   209   
2013-14                            204  204     204  204     204   204   
2014-15                            208  208     208  208     208   208   
2015-16                            215  215     215  215     215   215   
2016-17                            215  215     215  215     215   215   
2017-18                            210  210     210  210     210   210   
2018-19                            212  212     212  212     212   212   
2019-20                            217  217     217  217     217   217   
2020-21                            239  239     239  239     239   239   
2021-22                            217  217     217  217     217   217   
2022-23                            217  217     217  217     217   217   
2023-24                            214  214     214  214     214   214   
2024-25                            219  219     219  219     219   219   
2025-26                            230  230     230  230     230   230   

season_type_all_star          Regular Season                                  
measure_type_detailed_defense       Advanced Base Defense Misc Scoring Usage  
season                                                                        
2011-12                                  478  478     478  478     478   478  
2012-13                                  469  469     469  469     469   469  
2013-14                                  482  482     482  482     482   482  
2014-15                                  492  492     492  492     492   492  
2015-16                                  476  476     476  476     476   476  
2016-17                                  486  486     486  486     486   486  
2017-18                                  540  540     540  540     540   540  
2018-19                                  530  530     530  530     530   530  
2019-20                                  529  529     529  529     529   529  
2020-21                                  540  540     540  540     540   540  
2021-22                                  605  605     605  605     605   605  
2022-23                                  539  539     539  539     539   539  
2023-24                                  572  572     572  572     572   572  
2024-25                                  569  569     569  569     569   569  
2025-26                                  582  582     582  582     582   582

In [3]:
other = (log[log["endpoint"].isin(["LeagueDashPlayerShotLocations", "LeagueDashPlayerBioStats", "CommonTeamRoster"])]
         .assign(season_type=lambda d: d["season_type_all_star"].fillna("—"))
         .pivot_table(index="season", columns=["endpoint", "season_type"], values="rows", aggfunc="sum"))
other

endpoint    CommonTeamRoster LeagueDashPlayerBioStats  \
season_type                —                        —   
season                                                  
2011-12                  647                      478   
2012-13                  635                      469   
2013-14                  651                      482   
2014-15                  661                      492   
2015-16                  621                      476   
2016-17                  637                      486   
2017-18                  676                      540   
2018-19                  678                      530   
2019-20                  737                      529   
2020-21                  738                      540   
2021-22                  766                      605   
2022-23                  765                      539   
2023-24                  770                      572   
2024-25                  797                      569   
2025-26                  813                      582   

endpoint    LeagueDashPlayerShotLocations                 
season_type                      Playoffs Regular Season  
season                                                    
2011-12                               203            478  
2012-13                               209            469  
2013-14                               204            482  
2014-15                               208            492  
2015-16                               215            476  
2016-17                               215            486  
2017-18                               210            540  
2018-19                               212            530  
2019-20                               215            529  
2020-21                               239            540  
2021-22                               217            605  
2022-23                               217            539  
2023-24                               214            572  
2024-25                               219            569  
2025-26                               230            582

In [4]:
combine = log[log["endpoint"] == "DraftCombineStats"].set_index("season_all_time")["rows"]
print("combine players per draft year:\n", combine.to_string())
print("\nDraftHistory rows:", log.loc[log["endpoint"] == "DraftHistory", "rows"].item())

combine players per draft year:
 season_all_time
2000-01    65
2001-02    78
2002-03    82
2003-04    78
2004-05    81
2005-06    81
2006-07    76
2007-08    77
2008-09    76
2009-10    46
2010-11    52
2011-12    54
2012-13    61
2013-14    63
2014-15    56
2015-16    58
2016-17    56
2017-18    64
2018-19    69
2019-20    77
2020-21    45
2021-22    74
2022-23    83
2023-24    81
2024-25    83
2025-26    79
2026-27    78

DraftHistory rows: 8434


### Roster coverage (listed position, decision D4)

The roster row counts above include the coaches table. What matters is: for each season, what share of player-seasons (and of minutes) get a listed position from that season's rosters?

In [5]:
from nba_api.stats.endpoints import commonteamroster, leaguedashplayerstats
from nba_api.stats.static import teams
from unicorn.download import SEASONS
from unicorn.raw import fetch

def season_rosters(season):
    return pd.concat([fetch(commonteamroster.CommonTeamRoster, team_id=t["id"], season=season)["CommonTeamRoster"]
                      for t in teams.get_teams()], ignore_index=True)

def season_base(season):
    return fetch(leaguedashplayerstats.LeagueDashPlayerStats, season=season, season_type_all_star="Regular Season",
                 measure_type_detailed_defense="Base", per_mode_detailed="Totals")["LeagueDashPlayerStats"]

rows = []
for s in SEASONS:
    ros, base = season_rosters(s), season_base(s)
    has_pos = base["PLAYER_ID"].isin(ros.loc[ros["POSITION"].fillna("") != "", "PLAYER_ID"])
    rows.append({"season": s, "players": len(base), "roster_players": ros["PLAYER_ID"].nunique(),
                 "dup_roster_ids": ros["PLAYER_ID"].duplicated().sum(),
                 "pct_players_with_pos": has_pos.mean(),
                 "pct_minutes_with_pos": base.loc[has_pos, "MIN"].sum() / base["MIN"].sum()})
coverage = pd.DataFrame(rows).set_index("season")
coverage.style.format({"pct_players_with_pos": "{:.1%}", "pct_minutes_with_pos": "{:.1%}"})

,players,roster_players,dup_roster_ids,pct_players_with_pos,pct_minutes_with_pos
season,,,,,
2011-12,478,442,0,92.1%,98.7%
2012-13,469,446,0,93.0%,99.0%
2013-14,482,448,0,90.9%,98.8%
2014-15,492,450,0,90.9%,98.8%
2015-16,476,450,1,93.3%,98.7%
2016-17,486,451,0,91.8%,98.5%
2017-18,540,502,0,91.5%,98.8%
2018-19,530,497,0,91.1%,98.6%
2019-20,529,510,0,94.3%,99.2%


### Step 2A findings

- **All 703 requests are cached with no failures.** The six measure types agree exactly on player counts in every season and season type, so they can be joined one-to-one on `PLAYER_ID`.
- **Regular-season player counts** rise from about 470 (2011-14) to about 570 (2023-26). This reflects two-way contracts (from 2017-18) and larger rosters. **2021-22 has 605** because of COVID health-and-safety replacement signings.
- **Shot locations** occasionally have 1–2 fewer players than Base (2011-12 and 2019-20 playoffs): almost certainly players with zero FGA. We'll left-join and treat missing as 0 attempts.
- **Roster position coverage:** about 91% of player-seasons and about 99% of minutes. The low point is 2021-22 (81.5% of players, 97.5% of minutes), when the COVID call-ups had left rosters by the snapshot date. 2015-16 has one player on two rosters, so deduplicate when joining.
- **Combine:** 45–83 players per draft year. Dips in 2009-10 and 2020-21 (the 2020 combine was virtual).
- **Shortened seasons** to remember for counting stats: 2011-12 (66 games, lockout), 2019-20 (suspension + bubble), 2020-21 (72 games). Rate stats are unaffected; totals and games-played thresholds must be relative to team games.